<a href="https://colab.research.google.com/github/sway4em/566-cricket-colab/blob/new/pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Building the Best IPL Team
## Player Archetype Discovery via Unsupervised Learning

**Train:** 2008 - 2024 | **Test:** 2025 - 2026

### Setup
Upload your `data/` folder to Google Drive. Update `DATA_DIR` below to point to it.

In [ ]:
# Mount Google Drive and install dependencies
from google.colab import drive
drive.mount('/content/drive')

!pip install -q umap-learn

In [ ]:
import pandas as pd
import numpy as np
import os
import glob
from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score
from scipy import stats

import umap
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 50)

# UPDATE THIS PATH to where you uploaded the data/ folder in Google Drive
DATA_DIR = Path('/content/drive/MyDrive/566-term-project/data')

# Output directory for saved figures
OUTPUT_DIR = Path('/content/drive/MyDrive/566-term-project/outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Phase 1: Load and Combine Data

In [ ]:
csv_files = list(DATA_DIR.glob('*.csv'))
print(f"Found {len(csv_files)} match files")

dfs = []
for f in csv_files:
    df = pd.read_csv(f)
    dfs.append(df)

ball_data = pd.concat(dfs, ignore_index=True)
print(f"Total deliveries: {len(ball_data):,}")
print(f"Seasons (raw): {ball_data['season'].unique()}")
ball_data.head()

In [ ]:
# Normalize season labels
season_map = {
    '2007/08': '2008',
    '2009/10': '2010',
    '2020/21': '2020',
}
ball_data['season'] = ball_data['season'].astype(str).replace(season_map)
ball_data['season'] = ball_data['season'].astype(int)
print(f"Seasons after normalization: {sorted(ball_data['season'].unique())}")

In [ ]:
# Train/test split by season
TEST_SEASONS = [2025, 2026]
train_data = ball_data[~ball_data['season'].isin(TEST_SEASONS)].copy()
test_data = ball_data[ball_data['season'].isin(TEST_SEASONS)].copy()

print(f"Train: {len(train_data):,} deliveries ({train_data['season'].min()}-{train_data['season'].max()})")
print(f"Test:  {len(test_data):,} deliveries (seasons {TEST_SEASONS})")

## Phase 2: Feature Engineering

Compute per-player, per-season feature vectors. We define three game phases:
- **Powerplay:** Overs 1-6
- **Middle:** Overs 7-15
- **Death:** Overs 16-20

In [ ]:
def get_phase(ball_col):
    """Map ball number (e.g. 3.4 means over 4, ball 4) to game phase."""
    over = ball_col.astype(str).str.split('.').str[0].astype(int)
    phase = pd.Series('middle', index=ball_col.index)
    phase[over < 6] = 'powerplay'
    phase[over >= 15] = 'death'
    return phase

train_data['phase'] = get_phase(train_data['ball'])
test_data['phase'] = get_phase(test_data['ball'])

train_data['phase'].value_counts()

In [ ]:
def compute_batting_features(data):
    """Compute per-player, per-season batting features."""
    # Each row where the player is the striker counts as a ball faced
    # (unless it's a wide, which doesn't count as a ball faced)
    batting = data[data['wides'].isna() | (data['wides'] == 0)].copy()
    batting['is_boundary'] = batting['runs_off_bat'].isin([4, 6]).astype(int)
    batting['is_six'] = (batting['runs_off_bat'] == 6).astype(int)
    batting['is_dot'] = (batting['runs_off_bat'] == 0).astype(int)
    batting['is_dismissed'] = batting['player_dismissed'].notna().astype(int)

    group_cols = ['striker', 'season']

    # Overall stats
    overall = batting.groupby(group_cols).agg(
        total_runs=('runs_off_bat', 'sum'),
        balls_faced=('runs_off_bat', 'count'),
        boundaries=('is_boundary', 'sum'),
        sixes=('is_six', 'sum'),
        dots=('is_dot', 'sum'),
        dismissals=('is_dismissed', 'sum'),
    ).reset_index()

    overall['strike_rate'] = (overall['total_runs'] / overall['balls_faced']) * 100
    overall['boundary_pct'] = overall['boundaries'] / overall['balls_faced']
    overall['six_pct'] = overall['sixes'] / overall['balls_faced']
    overall['dot_pct'] = overall['dots'] / overall['balls_faced']
    overall['average'] = overall['total_runs'] / overall['dismissals'].clip(lower=1)

    # Phase-wise strike rates
    phase_sr = batting.groupby(group_cols + ['phase']).agg(
        phase_runs=('runs_off_bat', 'sum'),
        phase_balls=('runs_off_bat', 'count'),
        phase_dots=('is_dot', 'sum'),
        phase_boundaries=('is_boundary', 'sum'),
    ).reset_index()

    phase_sr['phase_sr'] = (phase_sr['phase_runs'] / phase_sr['phase_balls']) * 100
    phase_sr['phase_dot_pct'] = phase_sr['phase_dots'] / phase_sr['phase_balls']
    phase_sr['phase_boundary_pct'] = phase_sr['phase_boundaries'] / phase_sr['phase_balls']

    # Pivot phases into columns
    for metric in ['phase_sr', 'phase_dot_pct', 'phase_boundary_pct']:
        pivot = phase_sr.pivot_table(
            index=group_cols, columns='phase', values=metric
        ).reset_index()
        pivot.columns = [f'bat_{metric}_{c}' if c in ['powerplay', 'middle', 'death'] else c
                        for c in pivot.columns]
        overall = overall.merge(pivot, on=group_cols, how='left')

    # Consistency: std of runs per innings
    innings_runs = batting.groupby(['striker', 'season', 'match_id', 'innings']).agg(
        innings_runs=('runs_off_bat', 'sum'),
        innings_balls=('runs_off_bat', 'count'),
    ).reset_index()

    consistency = innings_runs.groupby(group_cols).agg(
        runs_std=('innings_runs', 'std'),
        avg_balls_per_innings=('innings_balls', 'mean'),
        num_innings=('innings_runs', 'count'),
    ).reset_index()

    overall = overall.merge(consistency, on=group_cols, how='left')

    # Acceleration: death SR / powerplay SR
    pp_col = 'bat_phase_sr_powerplay'
    death_col = 'bat_phase_sr_death'
    if pp_col in overall.columns and death_col in overall.columns:
        overall['acceleration'] = overall[death_col] / overall[pp_col].clip(lower=1)

    overall = overall.rename(columns={'striker': 'player'})
    return overall

bat_train = compute_batting_features(train_data)
bat_test = compute_batting_features(test_data)
print(f"Batting features: {bat_train.shape}")
bat_train.head()

In [ ]:
def compute_bowling_features(data):
    """Compute per-player, per-season bowling features."""
    bowling = data.copy()
    # Legitimate deliveries (not wides/noballs) for balls bowled count
    bowling['is_legit_delivery'] = (
        (bowling['wides'].isna() | (bowling['wides'] == 0)) &
        (bowling['noballs'].isna() | (bowling['noballs'] == 0))
    ).astype(int)
    bowling['runs_conceded'] = bowling['runs_off_bat'] + bowling['wides'].fillna(0) + bowling['noballs'].fillna(0)
    bowling['is_dot'] = ((bowling['runs_off_bat'] == 0) & (bowling['extras'].fillna(0) == 0)).astype(int)
    bowling['is_boundary_conceded'] = bowling['runs_off_bat'].isin([4, 6]).astype(int)
    bowling['is_wicket'] = (bowling['wicket_type'].notna() &
                            ~bowling['wicket_type'].isin(['run out', 'retired hurt', 'obstructing the field'])).astype(int)

    group_cols = ['bowler', 'season']

    overall = bowling.groupby(group_cols).agg(
        total_runs_conceded=('runs_conceded', 'sum'),
        balls_bowled=('is_legit_delivery', 'sum'),
        total_deliveries=('bowler', 'count'),
        wickets=('is_wicket', 'sum'),
        dots_bowled=('is_dot', 'sum'),
        boundaries_conceded=('is_boundary_conceded', 'sum'),
        extras_given=('extras', 'sum'),
    ).reset_index()

    overall['economy'] = (overall['total_runs_conceded'] / overall['balls_bowled']) * 6
    overall['bowling_sr'] = overall['balls_bowled'] / overall['wickets'].clip(lower=1)
    overall['dot_pct_bowl'] = overall['dots_bowled'] / overall['total_deliveries']
    overall['boundary_concede_pct'] = overall['boundaries_conceded'] / overall['total_deliveries']
    overall['extras_rate'] = overall['extras_given'] / overall['total_deliveries']

    # Phase-wise bowling
    phase_bowl = bowling.groupby(group_cols + ['phase']).agg(
        phase_runs_conceded=('runs_conceded', 'sum'),
        phase_balls=('is_legit_delivery', 'sum'),
        phase_wickets=('is_wicket', 'sum'),
        phase_dots=('is_dot', 'sum'),
        phase_deliveries=('bowler', 'count'),
    ).reset_index()

    phase_bowl['phase_economy'] = (phase_bowl['phase_runs_conceded'] / phase_bowl['phase_balls'].clip(lower=1)) * 6
    phase_bowl['phase_dot_pct'] = phase_bowl['phase_dots'] / phase_bowl['phase_deliveries']
    phase_bowl['phase_wicket_rate'] = phase_bowl['phase_wickets'] / phase_bowl['phase_balls'].clip(lower=1)

    for metric in ['phase_economy', 'phase_dot_pct', 'phase_wicket_rate']:
        pivot = phase_bowl.pivot_table(
            index=group_cols, columns='phase', values=metric
        ).reset_index()
        pivot.columns = [f'bowl_{metric}_{c}' if c in ['powerplay', 'middle', 'death'] else c
                        for c in pivot.columns]
        overall = overall.merge(pivot, on=group_cols, how='left')

    # Wicket type distribution
    wicket_types = bowling[bowling['is_wicket'] == 1].groupby(group_cols)['wicket_type'].value_counts().unstack(fill_value=0)
    if not wicket_types.empty:
        wicket_types = wicket_types.div(wicket_types.sum(axis=1), axis=0)
        wicket_types.columns = [f'wkt_pct_{c}' for c in wicket_types.columns]
        wicket_types = wicket_types.reset_index()
        overall = overall.merge(wicket_types, on=group_cols, how='left')

    overall = overall.rename(columns={'bowler': 'player'})
    return overall

bowl_train = compute_bowling_features(train_data)
bowl_test = compute_bowling_features(test_data)
print(f"Bowling features: {bowl_train.shape}")
bowl_train.head()

In [ ]:
def compute_batting_position_features(data):
    """Infer batting position from when a player first appears in each innings."""
    # Number deliveries sequentially within each innings
    data_sorted = data.sort_values(['match_id', 'innings', 'ball']).copy()
    data_sorted['delivery_num'] = data_sorted.groupby(['match_id', 'innings']).cumcount()

    # For each batter, find the delivery number they first appeared at in each innings
    first_appearance = data_sorted.groupby(['match_id', 'innings', 'striker'])['delivery_num'].min().reset_index()
    first_appearance.columns = ['match_id', 'innings', 'player', 'entry_delivery']

    # Merge season info
    match_seasons = data_sorted[['match_id', 'season']].drop_duplicates()
    first_appearance = first_appearance.merge(match_seasons, on='match_id')

    # Aggregate per player-season
    group_cols = ['player', 'season']
    position_features = first_appearance.groupby(group_cols).agg(
        avg_entry_delivery=('entry_delivery', 'mean'),
        median_entry_delivery=('entry_delivery', 'median'),
        entry_std=('entry_delivery', 'std'),
    ).reset_index()

    # What % of innings does the player open (entry at delivery 0)?
    opens = first_appearance[first_appearance['entry_delivery'] == 0].groupby(group_cols).size().reset_index(name='times_opened')
    total_innings = first_appearance.groupby(group_cols).size().reset_index(name='total_innings_pos')
    position_features = position_features.merge(opens, on=group_cols, how='left')
    position_features = position_features.merge(total_innings, on=group_cols, how='left')
    position_features['times_opened'] = position_features['times_opened'].fillna(0)
    position_features['opener_pct'] = position_features['times_opened'] / position_features['total_innings_pos']

    position_features = position_features.drop(columns=['times_opened', 'total_innings_pos'])
    return position_features

pos_train = compute_batting_position_features(train_data)
pos_test = compute_batting_position_features(test_data)

# Merge into batting features
bat_train = bat_train.merge(pos_train, on=['player', 'season'], how='left')
bat_test = bat_test.merge(pos_test, on=['player', 'season'], how='left')

print(f"Batting features with position: {bat_train.shape}")
# Sanity check
for p in ['CH Gayle', 'V Kohli', 'MS Dhoni', 'JJ Bumrah']:
    row = bat_train[bat_train['player'] == p]
    if len(row) > 0:
        print(f"  {p}: avg entry={row['avg_entry_delivery'].mean():.1f}, opener%={row['opener_pct'].mean():.1%}")

In [ ]:
def compute_bowling_phase_preference(data):
    """What % of a bowler's deliveries are in each phase? Captures role (PP specialist vs death bowler)."""
    phase_dist = data.groupby(['bowler', 'season', 'phase']).size().reset_index(name='deliveries')
    total = data.groupby(['bowler', 'season']).size().reset_index(name='total_deliveries')
    phase_dist = phase_dist.merge(total, on=['bowler', 'season'])
    phase_dist['phase_pct'] = phase_dist['deliveries'] / phase_dist['total_deliveries']

    pivot = phase_dist.pivot_table(index=['bowler', 'season'], columns='phase', values='phase_pct', fill_value=0).reset_index()
    pivot.columns = ['player', 'season', 'bowl_pct_death', 'bowl_pct_middle', 'bowl_pct_powerplay']
    return pivot

bowl_phase_train = compute_bowling_phase_preference(train_data)
bowl_phase_test = compute_bowling_phase_preference(test_data)

# Merge into bowling features
bowl_train = bowl_train.merge(bowl_phase_train, on=['player', 'season'], how='left')
bowl_test = bowl_test.merge(bowl_phase_test, on=['player', 'season'], how='left')

print(f"Bowling features with phase preference: {bowl_train.shape}")
for p in ['JJ Bumrah', 'R Ashwin', 'Rashid Khan']:
    row = bowl_train[bowl_train['player'] == p]
    if len(row) > 0:
        print(f"  {p}: PP={row['bowl_pct_powerplay'].mean():.0%}, Mid={row['bowl_pct_middle'].mean():.0%}, Death={row['bowl_pct_death'].mean():.0%}")

In [ ]:
# Merge batting and bowling features per player-season
# Players who both bat and bowl get full vectors; pure batters/bowlers get NaN for the other side

def merge_features(bat_df, bowl_df, min_balls_bat=30, min_balls_bowl=30):
    """Merge batting and bowling features with minimum threshold filtering."""
    bat_filtered = bat_df[bat_df['balls_faced'] >= min_balls_bat].copy()
    bowl_filtered = bowl_df[bowl_df['balls_bowled'] >= min_balls_bowl].copy()

    # Prefix columns to avoid collision
    bat_cols = [c for c in bat_filtered.columns if c not in ['player', 'season']]
    bowl_cols = [c for c in bowl_filtered.columns if c not in ['player', 'season']]

    bat_filtered = bat_filtered.rename(columns={c: f'bat_{c}' if not c.startswith('bat_') else c for c in bat_cols})
    bowl_filtered = bowl_filtered.rename(columns={c: f'bowl_{c}' if not c.startswith(('bowl_', 'wkt_')) else c for c in bowl_cols})

    merged = bat_filtered.merge(bowl_filtered, on=['player', 'season'], how='outer')

    # Label player type
    has_bat = merged['bat_balls_faced'].notna()
    has_bowl = merged['bowl_balls_bowled'].notna()
    merged['player_type'] = 'unknown'
    merged.loc[has_bat & ~has_bowl, 'player_type'] = 'batter'
    merged.loc[~has_bat & has_bowl, 'player_type'] = 'bowler'
    merged.loc[has_bat & has_bowl, 'player_type'] = 'allrounder'

    print(f"Player-seasons: {len(merged)}")
    print(merged['player_type'].value_counts())
    return merged

features_train = merge_features(bat_train, bowl_train)
features_test = merge_features(bat_test, bowl_test)
features_train.head()

## Phase 3: Dimensionality Reduction

In [ ]:
# Select numeric feature columns for reduction
exclude_cols = ['player', 'season', 'player_type']
feature_cols = [c for c in features_train.columns if c not in exclude_cols and features_train[c].dtype in ['float64', 'int64']]
print(f"Number of features: {len(feature_cols)}")
print(feature_cols)

In [ ]:
# Fill NaN with 0 for players who only bat or only bowl
X_train = features_train[feature_cols].fillna(0).values
X_test = features_test[feature_cols].fillna(0).values

# Standardize
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train matrix: {X_train_scaled.shape}")
print(f"Test matrix: {X_test_scaled.shape}")

In [ ]:
# PCA
pca = PCA(n_components=min(20, X_train_scaled.shape[1]))
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

# Variance explained
cumvar = np.cumsum(pca.explained_variance_ratio_)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(range(1, len(pca.explained_variance_ratio_) + 1), pca.explained_variance_ratio_)
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Variance Explained')
axes[0].set_title('Scree Plot')

axes[1].plot(range(1, len(cumvar) + 1), cumvar, 'bo-')
axes[1].axhline(y=0.9, color='r', linestyle='--', label='90% threshold')
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Variance Explained')
axes[1].set_title('Cumulative Variance')
axes[1].legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'pca_variance.png', dpi=150, bbox_inches='tight')
plt.show()

n_components_90 = np.argmax(cumvar >= 0.9) + 1
print(f"Components for 90% variance: {n_components_90}")

In [ ]:
# 2D PCA visualization colored by player type
fig, ax = plt.subplots(figsize=(10, 7))
colors = {'batter': 'blue', 'bowler': 'red', 'allrounder': 'green'}
for ptype, color in colors.items():
    mask = features_train['player_type'] == ptype
    ax.scatter(X_train_pca[mask, 0], X_train_pca[mask, 1],
              c=color, alpha=0.4, s=20, label=ptype)
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_title('PCA: Player-Season Vectors (2D)')
ax.legend()
plt.savefig(OUTPUT_DIR / 'pca_player_types.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# UMAP visualization
reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=30, min_dist=0.3)
X_train_umap = reducer.fit_transform(X_train_scaled)

fig, ax = plt.subplots(figsize=(10, 7))
for ptype, color in colors.items():
    mask = features_train['player_type'] == ptype
    ax.scatter(X_train_umap[mask, 0], X_train_umap[mask, 1],
              c=color, alpha=0.4, s=20, label=ptype)
ax.set_xlabel('UMAP-1')
ax.set_ylabel('UMAP-2')
ax.set_title('UMAP: Player-Season Vectors')
ax.legend()
plt.show()

## Phase 4: Clustering

In [ ]:
# Determine optimal k using silhouette scores
# Use PCA-reduced features (90% variance) for clustering
X_cluster = X_train_pca[:, :n_components_90]

k_range = range(3, 12)
silhouette_scores = []
inertias = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_cluster)
    silhouette_scores.append(silhouette_score(X_cluster, labels))
    inertias.append(km.inertia_)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(list(k_range), silhouette_scores, 'bo-')
axes[0].set_xlabel('k')
axes[0].set_ylabel('Silhouette Score')
axes[0].set_title('Silhouette Score vs k')

axes[1].plot(list(k_range), inertias, 'ro-')
axes[1].set_xlabel('k')
axes[1].set_ylabel('Inertia')
axes[1].set_title('Elbow Plot')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'cluster_selection.png', dpi=150, bbox_inches='tight')
plt.show()

best_k = list(k_range)[np.argmax(silhouette_scores)]
print(f"Best k by silhouette: {best_k} (score: {max(silhouette_scores):.3f})")

In [ ]:
# Fit final K-Means with best k
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
features_train['cluster_kmeans'] = kmeans.fit_predict(X_cluster)

# Assign test data to clusters
X_test_cluster = X_test_pca[:, :n_components_90]
features_test['cluster_kmeans'] = kmeans.predict(X_test_cluster)

print("Train cluster distribution:")
print(features_train['cluster_kmeans'].value_counts().sort_index())

In [ ]:
# GMM for soft clustering
gmm = GaussianMixture(n_components=best_k, random_state=42, covariance_type='full')
gmm.fit(X_cluster)
features_train['cluster_gmm'] = gmm.predict(X_cluster)
gmm_probs_train = gmm.predict_proba(X_cluster)

features_test['cluster_gmm'] = gmm.predict(X_test_cluster)
gmm_probs_test = gmm.predict_proba(X_test_cluster)

print(f"GMM BIC: {gmm.bic(X_cluster):.0f}")
print(f"GMM AIC: {gmm.aic(X_cluster):.0f}")

In [ ]:
# Visualize clusters in 2D (PCA and UMAP space)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

scatter1 = axes[0].scatter(X_train_pca[:, 0], X_train_pca[:, 1],
                           c=features_train['cluster_kmeans'], cmap='tab10', alpha=0.5, s=15)
axes[0].set_title('K-Means Clusters (PCA space)')
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')
plt.colorbar(scatter1, ax=axes[0])

scatter2 = axes[1].scatter(X_train_umap[:, 0], X_train_umap[:, 1],
                           c=features_train['cluster_kmeans'], cmap='tab10', alpha=0.5, s=15)
axes[1].set_title('K-Means Clusters (UMAP space)')
axes[1].set_xlabel('UMAP-1')
axes[1].set_ylabel('UMAP-2')
plt.colorbar(scatter2, ax=axes[1])

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'clusters_pca_umap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Interpret clusters: look at mean feature values per cluster
cluster_profiles = features_train.groupby('cluster_kmeans')[feature_cols].mean()

# Show key distinguishing features per cluster
key_bat_features = ['bat_strike_rate', 'bat_boundary_pct', 'bat_dot_pct', 'bat_average',
                    'bat_phase_sr_powerplay', 'bat_phase_sr_death', 'acceleration',
                    'bat_avg_entry_delivery', 'bat_opener_pct']
key_bowl_features = ['bowl_economy', 'bowl_bowling_sr', 'bowl_dot_pct_bowl',
                     'bowl_phase_economy_powerplay', 'bowl_phase_economy_death']

key_features = [f for f in key_bat_features + key_bowl_features if f in feature_cols]
print("Cluster profiles (key features):")
cluster_profiles[key_features].round(2)

In [ ]:
# Spot-check: look at well-known players in each cluster
known_players = ['V Kohli', 'MS Dhoni', 'AB de Villiers', 'JJ Bumrah',
                 'Rashid Khan', 'DJ Bravo', 'SP Narine', 'DA Warner',
                 'KL Rahul', 'RA Jadeja', 'CH Gayle', 'YS Chahal']

player_clusters = features_train[features_train['player'].isin(known_players)][['player', 'season', 'player_type', 'cluster_kmeans']]
player_clusters.sort_values(['player', 'season'])

## Phase 5: Team Composition Analysis

Compare archetype distributions of playoff vs non-playoff teams.

In [ ]:
# We need to map players to teams per season
# Extract from the ball data: a player's team is who they bat/bowl for

def get_player_teams(data):
    """Get player-team-season mapping from ball data."""
    bat_teams = data[['striker', 'batting_team', 'season']].rename(
        columns={'striker': 'player', 'batting_team': 'team'})
    bowl_teams = data[['bowler', 'bowling_team', 'season']].rename(
        columns={'bowler': 'player', 'bowling_team': 'team'})
    all_teams = pd.concat([bat_teams, bowl_teams]).drop_duplicates()
    # Take most common team per player-season (handles edge cases)
    player_teams = all_teams.groupby(['player', 'season'])['team'].agg(
        lambda x: x.value_counts().index[0]
    ).reset_index()
    return player_teams

player_teams_train = get_player_teams(train_data)
player_teams_test = get_player_teams(test_data)

# Merge with features
features_train = features_train.merge(player_teams_train, on=['player', 'season'], how='left')
features_test = features_test.merge(player_teams_test, on=['player', 'season'], how='left')

print(f"Teams in training data: {features_train['team'].nunique()}")
features_train[['player', 'season', 'team', 'cluster_kmeans']].head(10)

In [ ]:
# IPL Playoff teams by season (top 4)
playoff_teams = {
    2008: ['Rajasthan Royals', 'Chennai Super Kings', 'Delhi Daredevils', 'Kings XI Punjab'],
    2009: ['Deccan Chargers', 'Royal Challengers Bangalore', 'Delhi Daredevils', 'Chennai Super Kings'],
    2010: ['Chennai Super Kings', 'Mumbai Indians', 'Royal Challengers Bangalore', 'Deccan Chargers'],
    2011: ['Chennai Super Kings', 'Royal Challengers Bangalore', 'Mumbai Indians', 'Kolkata Knight Riders'],
    2012: ['Kolkata Knight Riders', 'Chennai Super Kings', 'Delhi Daredevils', 'Mumbai Indians'],
    2013: ['Mumbai Indians', 'Chennai Super Kings', 'Rajasthan Royals', 'Sunrisers Hyderabad'],
    2014: ['Kolkata Knight Riders', 'Kings XI Punjab', 'Chennai Super Kings', 'Mumbai Indians'],
    2015: ['Mumbai Indians', 'Chennai Super Kings', 'Royal Challengers Bangalore', 'Rajasthan Royals'],
    2016: ['Sunrisers Hyderabad', 'Royal Challengers Bangalore', 'Gujarat Lions', 'Kolkata Knight Riders'],
    2017: ['Mumbai Indians', 'Rising Pune Supergiant', 'Sunrisers Hyderabad', 'Kolkata Knight Riders'],
    2018: ['Chennai Super Kings', 'Sunrisers Hyderabad', 'Kolkata Knight Riders', 'Rajasthan Royals'],
    2019: ['Mumbai Indians', 'Chennai Super Kings', 'Delhi Capitals', 'Sunrisers Hyderabad'],
    2020: ['Mumbai Indians', 'Delhi Capitals', 'Sunrisers Hyderabad', 'Royal Challengers Bangalore'],
    2021: ['Chennai Super Kings', 'Kolkata Knight Riders', 'Delhi Capitals', 'Royal Challengers Bangalore'],
    2022: ['Gujarat Titans', 'Rajasthan Royals', 'Lucknow Super Giants', 'Royal Challengers Bangalore'],
    2023: ['Chennai Super Kings', 'Gujarat Titans', 'Mumbai Indians', 'Lucknow Super Giants'],
    2024: ['Kolkata Knight Riders', 'Sunrisers Hyderabad', 'Rajasthan Royals', 'Royal Challengers Bengaluru'],
}

# Label each team-season as playoff or not
def is_playoff(row):
    season = row['season']
    team = row['team']
    if season in playoff_teams:
        return team in playoff_teams[season]
    return None

features_train['is_playoff'] = features_train.apply(is_playoff, axis=1)
print(features_train['is_playoff'].value_counts(dropna=False))

In [ ]:
# Compute archetype distribution per team-season
team_compositions = features_train.groupby(['team', 'season', 'is_playoff'])['cluster_kmeans'].value_counts(
    normalize=True
).unstack(fill_value=0).reset_index()

team_compositions.columns.name = None
cluster_cols = [c for c in team_compositions.columns if isinstance(c, (int, np.integer))]

print(f"Team-season compositions: {len(team_compositions)}")
team_compositions.head()

In [ ]:
# Compare playoff vs non-playoff archetype distributions
playoff_comp = team_compositions[team_compositions['is_playoff'] == True][cluster_cols].mean()
non_playoff_comp = team_compositions[team_compositions['is_playoff'] == False][cluster_cols].mean()

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(cluster_cols))
width = 0.35

ax.bar(x - width/2, playoff_comp.values, width, label='Playoff Teams', color='green', alpha=0.7)
ax.bar(x + width/2, non_playoff_comp.values, width, label='Non-Playoff Teams', color='red', alpha=0.7)
ax.set_xlabel('Cluster (Archetype)')
ax.set_ylabel('Proportion of Squad')
ax.set_title('Archetype Distribution: Playoff vs Non-Playoff Teams')
ax.set_xticks(x)
ax.set_xticklabels([f'C{c}' for c in cluster_cols])
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'playoff_vs_nonplayoff.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Statistical test: are the distributions significantly different?
print("Per-archetype comparison (playoff vs non-playoff):")
print(f"{'Cluster':<10} {'Playoff Mean':<15} {'Non-PO Mean':<15} {'t-stat':<10} {'p-value':<10}")
print("-" * 60)

for c in cluster_cols:
    po_vals = team_compositions[team_compositions['is_playoff'] == True][c]
    npo_vals = team_compositions[team_compositions['is_playoff'] == False][c]
    t_stat, p_val = stats.ttest_ind(po_vals, npo_vals)
    sig = '*' if p_val < 0.05 else ''
    print(f"C{c:<9} {po_vals.mean():<15.3f} {npo_vals.mean():<15.3f} {t_stat:<10.3f} {p_val:<10.4f} {sig}")

## Phase 6: Replacement Recommender

In [ ]:
def find_replacement(player_name, season, features_df, pca_features, top_n=5, exclude_team=None):
    """
    Find the most similar players to a given player based on their feature vector.
    Uses the most recent season's data for the target player.
    """
    # Get target player's vector
    target_mask = (features_df['player'] == player_name) & (features_df['season'] == season)
    if target_mask.sum() == 0:
        # Try most recent season
        player_data = features_df[features_df['player'] == player_name]
        if len(player_data) == 0:
            print(f"Player '{player_name}' not found.")
            return None
        season = player_data['season'].max()
        target_mask = (features_df['player'] == player_name) & (features_df['season'] == season)

    target_idx = features_df[target_mask].index[0]
    target_vec = pca_features[features_df.index.get_loc(target_idx)]

    # Compute distances to all other players (use their most recent season)
    latest_season_per_player = features_df.groupby('player')['season'].max().reset_index()
    latest_season_per_player.columns = ['player', 'latest_season']

    candidates = features_df.merge(latest_season_per_player, on='player')
    candidates = candidates[candidates['season'] == candidates['latest_season']]
    candidates = candidates[candidates['player'] != player_name]

    if exclude_team:
        candidates = candidates[candidates['team'] != exclude_team]

    candidate_indices = [features_df.index.get_loc(idx) for idx in candidates.index]
    candidate_vecs = pca_features[candidate_indices]

    distances = np.linalg.norm(candidate_vecs - target_vec, axis=1)
    candidates = candidates.copy()
    candidates['distance'] = distances

    result = candidates.nsmallest(top_n, 'distance')[['player', 'season', 'team', 'cluster_kmeans', 'player_type', 'distance']]
    print(f"\nTop {top_n} replacements for {player_name} (season {season}):")
    return result

# Example: find replacements for a departing player
find_replacement('MS Dhoni', 2024, features_train, X_cluster)

In [ ]:
find_replacement('JJ Bumrah', 2024, features_train, X_cluster)

## Phase 7: Validation on Test Set (2025-2026)

In [ ]:
# Check if clusters are stable: do players keep similar cluster assignments across seasons?
# For players appearing in both train and test

train_latest = features_train.groupby('player').apply(
    lambda x: x.loc[x['season'].idxmax()]
).reset_index(drop=True)[['player', 'cluster_kmeans']].rename(columns={'cluster_kmeans': 'cluster_train'})

test_clusters = features_test[['player', 'season', 'cluster_kmeans']].rename(columns={'cluster_kmeans': 'cluster_test'})

overlap = test_clusters.merge(train_latest, on='player')
overlap['same_cluster'] = overlap['cluster_train'] == overlap['cluster_test']

stability = overlap['same_cluster'].mean()
print(f"Players in both train & test: {len(overlap)}")
print(f"Cluster stability (same cluster): {stability:.1%}")
print(f"\nCluster transition matrix:")
pd.crosstab(overlap['cluster_train'], overlap['cluster_test'], margins=True)

In [ ]:
# Validate team composition patterns on test seasons
# Do playoff teams in 2025/2026 follow the same archetype patterns?

# Playoff teams for test seasons (derived from match data - playoff matches at end of season)
# 2025: Final RCB vs PBKS (Jun 3), playoffs involved MI, GT, PBKS, RCB
# 2026: Final GT vs RCB (May 31), playoffs involved GT, RCB, RR, SRH
playoff_teams_test = {
    2025: ['Mumbai Indians', 'Gujarat Titans', 'Punjab Kings', 'Royal Challengers Bengaluru'],
    2026: ['Gujarat Titans', 'Royal Challengers Bengaluru', 'Rajasthan Royals', 'Sunrisers Hyderabad'],
}

features_test['is_playoff'] = features_test.apply(
    lambda r: r['team'] in playoff_teams_test.get(r['season'], []), axis=1)

test_compositions = features_test.groupby(['team', 'season', 'is_playoff'])['cluster_kmeans'].value_counts(
    normalize=True).unstack(fill_value=0).reset_index()
test_compositions.columns.name = None

# Compare test playoff vs non-playoff (same analysis as training)
test_cluster_cols = [c for c in test_compositions.columns if isinstance(c, (int, np.integer))]
test_po = test_compositions[test_compositions['is_playoff'] == True][test_cluster_cols].mean()
test_npo = test_compositions[test_compositions['is_playoff'] == False][test_cluster_cols].mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training pattern
axes[0].bar(np.arange(len(cluster_cols)) - 0.175, playoff_comp.values, 0.35, label='Playoff', color='green', alpha=0.7)
axes[0].bar(np.arange(len(cluster_cols)) + 0.175, non_playoff_comp.values, 0.35, label='Non-Playoff', color='red', alpha=0.7)
axes[0].set_title('Train (2008-2024)')
axes[0].set_xlabel('Archetype Cluster')
axes[0].set_ylabel('Proportion')
axes[0].legend()

# Test pattern
axes[1].bar(np.arange(len(test_cluster_cols)) - 0.175, test_po.values, 0.35, label='Playoff', color='green', alpha=0.7)
axes[1].bar(np.arange(len(test_cluster_cols)) + 0.175, test_npo.values, 0.35, label='Non-Playoff', color='red', alpha=0.7)
axes[1].set_title('Test (2025-2026)')
axes[1].set_xlabel('Archetype Cluster')
axes[1].set_ylabel('Proportion')
axes[1].legend()

plt.suptitle('Do playoff team composition patterns generalize?')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'train_vs_test_validation.png', dpi=150, bbox_inches='tight')
plt.show()

## Phase 8: Structural Validation — Do Teams Have Distinct Roles?

If the clustering is meaningful, we should see that teams DON'T have lots of duplicate players.
Each team should fill specific roles: ~2 openers, ~2 middle-order anchors, ~2 finishers, ~2 death bowlers, etc.
We validate by checking:
1. Do all teams have similar archetype counts? (low variance across teams = universal team structure)
2. Is within-team player diversity high? (teammates should be in DIFFERENT clusters)
3. Can we find the "2 openers per team" pattern without telling the model about it?

In [ ]:
# 1. How many players per cluster does each team have? (per season)
team_cluster_counts = features_train.groupby(['team', 'season', 'cluster_kmeans']).size().reset_index(name='count')

# Average across all team-seasons: how many players of each archetype?
avg_per_cluster = team_cluster_counts.groupby('cluster_kmeans')['count'].agg(['mean', 'std']).round(2)
avg_per_cluster.columns = ['avg_players', 'std_players']
print("Average players per archetype per team-season:")
print(avg_per_cluster)
print(f"\nLow std = all teams fill this role similarly (universal team structure)")

# Visualize: distribution of counts per cluster across all team-seasons
fig, ax = plt.subplots(figsize=(10, 5))
team_cluster_counts.boxplot(column='count', by='cluster_kmeans', ax=ax)
ax.set_xlabel('Cluster (Archetype)')
ax.set_ylabel('Number of Players per Team')
ax.set_title('How Many Players Per Archetype Does Each Team Have?')
plt.suptitle('')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'team_role_counts.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 2. Within-team diversity: are teammates spread across clusters or clumped?
# Measure: for each team-season, what's the entropy of cluster assignments?
from scipy.stats import entropy

def team_diversity(group):
    counts = group['cluster_kmeans'].value_counts(normalize=True)
    return entropy(counts)

team_entropy = features_train.groupby(['team', 'season']).apply(team_diversity).reset_index(name='diversity')

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(team_entropy['diversity'], bins=20, edgecolor='black', alpha=0.7)
ax.axvline(team_entropy['diversity'].mean(), color='red', linestyle='--', label=f"Mean: {team_entropy['diversity'].mean():.2f}")
ax.axvline(np.log(best_k), color='green', linestyle='--', label=f"Max possible (uniform): {np.log(best_k):.2f}")
ax.set_xlabel('Shannon Entropy of Cluster Distribution')
ax.set_ylabel('Number of Team-Seasons')
ax.set_title('Within-Team Archetype Diversity\n(Higher = more diverse roster)')
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'team_diversity.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Mean diversity: {team_entropy['diversity'].mean():.3f} / max {np.log(best_k):.3f}")
print("Teams close to max entropy = they fill all roles rather than stacking one type")

In [ ]:
# 3. Cross-team role matching: do teams share a similar "skeleton"?
# For each cluster, show the raw count of players per team-season as a heatmap (one recent season)
recent_season = 2024
recent_teams = features_train[features_train['season'] == recent_season]

if len(recent_teams) > 0:
    team_cluster_matrix = recent_teams.groupby(['team', 'cluster_kmeans']).size().unstack(fill_value=0)

    fig, ax = plt.subplots(figsize=(10, 6))
    sns.heatmap(team_cluster_matrix, annot=True, fmt='d', cmap='YlOrRd', ax=ax)
    ax.set_xlabel('Archetype Cluster')
    ax.set_ylabel('Team')
    ax.set_title(f'Players Per Archetype by Team ({recent_season})\nSimilar rows = universal team structure')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'team_archetype_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()

    print("If all rows look similar, teams build rosters with the same archetype mix.")
    print("This confirms the model captures real positional roles.")

In [ ]:
# 4. Feature importance: which features drive the principal components?
# Great for the "feature engineering" section of your presentation

# Top features contributing to PC1 and PC2
loadings = pd.DataFrame(
    pca.components_[:3].T,
    columns=['PC1', 'PC2', 'PC3'],
    index=feature_cols
)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# PC1 top features
top_pc1 = loadings['PC1'].abs().nlargest(10)
axes[0].barh(range(len(top_pc1)), loadings.loc[top_pc1.index, 'PC1'].values)
axes[0].set_yticks(range(len(top_pc1)))
axes[0].set_yticklabels(top_pc1.index, fontsize=9)
axes[0].set_xlabel('Loading')
axes[0].set_title('Top 10 Features Driving PC1')

# PC2 top features
top_pc2 = loadings['PC2'].abs().nlargest(10)
axes[1].barh(range(len(top_pc2)), loadings.loc[top_pc2.index, 'PC2'].values)
axes[1].set_yticks(range(len(top_pc2)))
axes[1].set_yticklabels(top_pc2.index, fontsize=9)
axes[1].set_xlabel('Loading')
axes[1].set_title('Top 10 Features Driving PC2')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'pca_loadings.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 5. Cluster radar chart: visualize what each archetype "looks like"
# Normalize key features to 0-1 for radar plot comparison

from matplotlib.patches import FancyBboxPatch

radar_features = [f for f in ['bat_strike_rate', 'bat_boundary_pct', 'bat_dot_pct',
                               'bat_avg_entry_delivery', 'bat_opener_pct',
                               'bowl_economy', 'bowl_dot_pct_bowl',
                               'bowl_pct_powerplay', 'bowl_pct_death'] if f in feature_cols]

if len(radar_features) >= 4:
    profiles = features_train.groupby('cluster_kmeans')[radar_features].mean()
    # Normalize each feature to 0-1 range
    profiles_norm = (profiles - profiles.min()) / (profiles.max() - profiles.min() + 1e-10)

    angles = np.linspace(0, 2 * np.pi, len(radar_features), endpoint=False).tolist()
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
    colors_radar = plt.cm.tab10(np.linspace(0, 1, best_k))

    for i, (cluster_id, row) in enumerate(profiles_norm.iterrows()):
        values = row.tolist()
        values += values[:1]
        ax.plot(angles, values, 'o-', linewidth=2, label=f'Cluster {cluster_id}', color=colors_radar[i])
        ax.fill(angles, values, alpha=0.1, color=colors_radar[i])

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels([f.replace('bat_', '').replace('bowl_', '') for f in radar_features], size=9)
    ax.set_title('Archetype Profiles (Radar Chart)', size=14, pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'cluster_radar.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("Not enough radar features available")

In [ ]:
# 6. Named examples per cluster: helps presentation narrative
# Show top 3 most "representative" players per cluster (closest to centroid)

print("Most representative players per archetype:\n")
for c in range(best_k):
    mask = features_train['cluster_kmeans'] == c
    cluster_data = features_train[mask]
    centroid = kmeans.cluster_centers_[c]
    cluster_pca = X_cluster[mask.values]
    dists = np.linalg.norm(cluster_pca - centroid, axis=1)
    closest_idx = np.argsort(dists)[:5]
    closest_players = cluster_data.iloc[closest_idx][['player', 'season', 'player_type']].values
    print(f"Cluster {c}:")
    for player, season, ptype in closest_players:
        print(f"  {player} ({int(season)}) - {ptype}")
    print()

In [ ]:
# Replacement recommender validation:
# For players who changed teams between 2024 and 2025, check if the acquiring team
# got a player from a similar cluster to someone they lost

players_2024 = features_train[features_train['season'] == 2024][['player', 'team', 'cluster_kmeans']]
players_2025 = features_test[features_test['season'] == 2025][['player', 'team', 'cluster_kmeans']]

# Players who moved teams
moved = players_2024.merge(players_2025, on='player', suffixes=('_2024', '_2025'))
moved = moved[moved['team_2024'] != moved['team_2025']]
print(f"Players who changed teams: {len(moved)}")
if len(moved) > 0:
    print(moved[['player', 'team_2024', 'team_2025', 'cluster_kmeans_2024', 'cluster_kmeans_2025']].head(15))

# Save all computed data to Drive
features_train.to_csv(OUTPUT_DIR / 'features_train.csv', index=False)
features_test.to_csv(OUTPUT_DIR / 'features_test.csv', index=False)
team_compositions.to_csv(OUTPUT_DIR / 'team_compositions_train.csv', index=False)
test_compositions.to_csv(OUTPUT_DIR / 'team_compositions_test.csv', index=False)

# Save cluster profiles
cluster_profiles = features_train.groupby('cluster_kmeans')[feature_cols].mean()
cluster_profiles.to_csv(OUTPUT_DIR / 'cluster_profiles.csv')

print(f"All outputs saved to: {OUTPUT_DIR}")
print(f"\nFigures saved:")
for f in sorted(OUTPUT_DIR.glob('*.png')):
    print(f"  {f.name}")
print(f"\nData saved:")
for f in sorted(OUTPUT_DIR.glob('*.csv')):
    print(f"  {f.name}")

In [ ]:
def search_player(query):
    """Search for players by partial name (case-insensitive).
    Example: search_player('bumrah') -> ['JJ Bumrah']
    """
    all_players = sorted(features_train['player'].unique())
    matches = [p for p in all_players if query.lower() in p.lower()]
    if not matches:
        print(f"No players found matching '{query}'")
    else:
        print(f"Found {len(matches)} match(es) for '{query}':")
        for p in matches:
            seasons = sorted(features_train[features_train['player'] == p]['season'].unique())
            print(f"  {p}  (seasons: {seasons[0]}-{seasons[-1]})")
    return matches

# Examples:
search_player('bumrah')
search_player('kohli')
search_player('dhoni')

## Summary & Next Steps

Key outputs:
1. **Player archetypes** discovered via unsupervised clustering on engineered features
2. **Team composition patterns** comparing playoff vs non-playoff squads
3. **Replacement recommender** using feature-space distance
4. **Validation** on held-out 2025/2026 seasons

Potential extensions:
- HDBSCAN for density-based clustering (handles outliers better)
- Temporal analysis: how do player archetypes evolve over careers?
- Auction price analysis: are certain archetypes overvalued?
- Match outcome prediction from team archetype composition